In [19]:
import pandas as pd 
import numpy as np

In [20]:
df = pd.read_csv("hea_dataset.csv")
df

,composition,structure,total_magnetization
0,Al2Fe9Co9(SiNi3)3,Full Formula (Al2 Fe9 Co9 Si3 Ni9)\nReduced Fo...,36.998
1,Al7Fe6Co7(CuNi)6,Full Formula (Al7 Fe6 Co7 Cu6 Ni6)\nReduced Fo...,24.329
2,MnFe10Co10SiNi10,Full Formula (Mn1 Fe10 Co10 Si1 Ni10)\nReduced...,42.226
3,Cr2Fe10Co9Si2Ni9,Full Formula (Cr2 Fe10 Co9 Si2 Ni9)\nReduced F...,36.040
4,AlFe10Co10SiNi10,Full Formula (Al1 Fe10 Co10 Si1 Ni10)\nReduced...,45.880
5,Ti2Fe10Co9Cu2Ni9,Full Formula (Ti2 Fe10 Co9 Cu2 Ni9)\nReduced F...,41.953
6,Mn2Fe10Co9Si2Ni9,Full Formula (Mn2 Fe10 Co9 Si2 Ni9)\nReduced F...,35.742
7,Al5Fe4Co5(Cu2Ni7)2,Full Formula (Al5 Fe4 Co5 Cu4 Ni14)\nReduced F...,21.926
8,Mn2Fe9Co9(SiNi3)3,Full Formula (Mn2 Fe9 Co9 Si3 Ni9)\nReduced Fo...,30.932
9,AlFe15Co8SiNi7,Full Formula (Al1 Fe15 Co8 Si1 Ni7)\nReduced F...,54.801


In [28]:
#!/usr/bin/env python3
"""
Process HEA dataset CSV containing structure text (printed pymatgen.Structure)
and produce a neighbor-shell CSV.

Dependencies:
    pip install pandas pymatgen numpy

Usage (in script):
    Adjust INPUT_CSV if needed, then run the script.

Notes:
 - This code attempts to handle three structure formats commonly seen in HEA CSVs:
   1) printed pymatgen.Structure (the "Full Formula / Sites" text you pasted),
   2) JSON-serialized Structure (Structure.as_dict()),
   3) CIF file paths or raw CIF text (fallback).
 - It outputs one row per site in each structure. If many structures exist this can
   produce a large file.
"""

import re
import json
import tempfile
from pathlib import Path
import pandas as pd
import numpy as np
from pymatgen.core import Lattice, Structure

# ----------------- CONFIG -----------------
INPUT_CSV = "hea_dataset.csv"   # change if your file is elsewhere
OUTPUT_CSV = "candidate_neighbor_configurations_from_heas.csv"

MAX_SHELLS = 4
MAX_NEIGHBOR_DIST = 6.0     # Å — tune for your chemistries
DISTANCE_TOLERANCE = 0.1  # Å: max gap inside same shell
# ------------------------------------------

float_re = re.compile(r"[-+]?\d*\.\d+|[-+]?\d+")


def detect_structure_column(df: pd.DataFrame):
    # Prefer obvious names
    for candidate in ["structure", "struct", "structure_str", "structure_text", "cif"]:
        if candidate in df.columns:
            return candidate
    # fallback: longest text column
    lengths = [(c, df[c].astype(str).str.len().median()) for c in df.columns]
    lengths = sorted(lengths, key=lambda x: (0 if np.isnan(x[1]) else x[1]), reverse=True)
    return lengths[0][0]


def write_temp_file(text: str, suffix=".cif"):
    tf = tempfile.NamedTemporaryFile(delete=False, suffix=suffix, mode="w", encoding="utf-8")
    tf.write(text)
    tf.flush()
    tf.close()
    return Path(tf.name)


def parse_printed_structure(s: str) -> Structure:
    """
    Parse the printed pymatgen.Structure representation (the 'Full Formula / Sites' block).
    Robustly extracts abc, angles and site list.
    """
    lines = [ln.rstrip() for ln in s.splitlines() if ln.strip() != ""]
    abc = None
    angles = None

    # find abc and angles lines (e.g., "abc   :   7.265653   7.054177   7.048887")
    for ln in lines:
        ln_strip = ln.strip()
        if ln_strip.lower().startswith("abc"):
            nums = float_re.findall(ln_strip)
            if len(nums) >= 3:
                abc = [float(nums[0]), float(nums[1]), float(nums[2])]
        if ln_strip.lower().startswith("angles"):
            nums = float_re.findall(ln_strip)
            if len(nums) >= 3:
                angles = [float(nums[0]), float(nums[1]), float(nums[2])]

    if abc is None or angles is None:
        raise ValueError("Could not parse lattice parameters (abc/angles) from printed structure.")

    # find the start of the sites table: a line that starts with '#' or contains "Sites ("
    start_idx = None
    for i, ln in enumerate(lines):
        if ln.strip().startswith("#") or ln.lower().startswith("sites"):
            start_idx = i + 1
            break
    if start_idx is None:
        # maybe the table header appears later; fallback: find line containing 'SP' or 'a' 'b' 'c' tokens
        for i, ln in enumerate(lines):
            if re.search(r"\bSP\b", ln) and re.search(r"\ba\b", ln) and re.search(r"\bb\b", ln):
                start_idx = i + 1
                break

    if start_idx is None:
        raise ValueError("Could not find site table in printed structure.")

    species = []
    coords = []
    # iterate remaining lines and extract species + first three floats per line
    for ln in lines[start_idx:]:
        ln_strip = ln.strip()
        # skip separators (---)
        if set(ln_strip) <= set("- ") or ln_strip.startswith("----") or "----" in ln_strip:
            continue
        if ln_strip == "":
            continue

        tokens = ln_strip.split()
        if len(tokens) < 2:
            continue

        # Species token heuristic: first token that contains letters (e.g., 'Al', 'Fe', 'Ni3' etc.)
        sp_token = None
        for t in tokens[:3]:
            if re.search(r"[A-Za-z]", t):
                sp_token = t
                break
        if sp_token is None:
            # fallback to token index 1 (common format '# idx SP x y z')
            sp_token = tokens[1] if len(tokens) > 1 else tokens[0]

        # find floats in the line
        nums = float_re.findall(ln_strip)
        if len(nums) < 3:
            # skip lines that don't have coordinate numbers
            continue

        # take the first three numeric tokens as fractional coords
        x, y, z = float(nums[0]), float(nums[1]), float(nums[2])
        species.append(sp_token)
        coords.append([x, y, z])

    if len(species) == 0:
        raise ValueError("No atomic sites parsed from printed structure.")

    lattice = Lattice.from_parameters(abc[0], abc[1], abc[2], angles[0], angles[1], angles[2])
    structure = Structure(lattice, species, coords)
    return structure


def parse_structure_cell(cell_value, base_dir: Path):
    """
    Given the content of the structure cell, attempt robust parsing:
     - JSON dict (pymatgen Structure.as_dict())
     - printed structure (the block you pasted)
     - CIF path or raw CIF text (fallback)
    Returns a pymatgen Structure instance.
    """
    if pd.isna(cell_value):
        raise ValueError("Empty structure cell.")

    s = str(cell_value).strip()

    # 1) JSON-like Structure dictionary
    if s.startswith("{") and ("@module" in s or '"lattice"' in s):
        try:
            d = json.loads(s)
            return Structure.from_dict(d)
        except Exception:
            # try load loosely: sometimes single-quotes -> replace with double quotes (dangerous but occasionally works)
            try:
                d = json.loads(s.replace("'", '"'))
                return Structure.from_dict(d)
            except Exception:
                pass

    # 2) If it looks like a path to a cif file
    if s.lower().endswith(".cif") or "/" in s or "\\" in s:
        p = Path(s)
        if not p.exists():
            p2 = (base_dir / s).resolve()
            if p2.exists():
                p = p2
        if p.exists():
            # use pymatgen's file reader
            return Structure.from_file(str(p))

    # 3) Raw CIF text?
    if "data_" in s or "_atom_site" in s or "loop_" in s:
        tf = write_temp_file(s, suffix=".cif")
        try:
            struct = Structure.from_file(str(tf))
            return struct
        finally:
            try:
                tf.unlink()
            except Exception:
                pass

    # 4) Printed structure (Full Formula / Sites)
    # Heuristic: contains 'Full Formula' or 'Sites (' or 'abc' and 'angles'
    if ("Full Formula" in s) or ("Sites (" in s) or ("abc" in s and "angles" in s):
        return parse_printed_structure(s)

    # 5) POSCAR-like? (starts with element or scale factor)
    # try Structure.from_str with 'poscar' format (pymatgen supports parsing by file extension; we can try a temp file)
    # fallback: write to temp file and attempt Structure.from_file (pymatgen guesses by extension if .vasp, .cif etc)
    # but avoid infinite guesses — raise for now
    raise ValueError("Unrecognized structure format (not JSON, not printed structure, not CIF path).")


def group_into_shells(neighbor_data, tol=DISTANCE_TOLERANCE):
    if not neighbor_data:
        return []

    neighbor_data = sorted(neighbor_data, key=lambda x: x["distance"])
    shells = []
    current = [neighbor_data[0]]

    for nd in neighbor_data[1:]:
        prev = current[-1]
        if (nd["distance"] - prev["distance"]) <= tol:
            current.append(nd)
        else:
            shells.append(current)
            current = [nd]
    if current:
        shells.append(current)
    return shells


def extract_environment(structure: Structure, central_index: int,
                        max_dist=MAX_NEIGHBOR_DIST, max_shells=MAX_SHELLS):
    central_site = structure[central_index]
    # pymatgen's get_neighbors is periodic-aware
    neighbors = structure.get_neighbors(central_site, max_dist)
    neighbor_data = []
    for nb in neighbors:
        d = float(central_site.distance(nb))
        neighbor_data.append({"distance": d, "species": nb.species_string})
    if not neighbor_data:
        return []
    neighbor_data.sort(key=lambda x: x["distance"])
    shells_raw = group_into_shells(neighbor_data, tol=DISTANCE_TOLERANCE)
    shell_info = []
    for shell in shells_raw[:max_shells]:
        species_set = set(n["species"] for n in shell)
        shell_info.append({
            "count": len(shell),
            "n_species": len(species_set),
            "species": ",".join(sorted(species_set)),
            "distances": [n["distance"] for n in shell]
        })
    return shell_info


def build_neighbor_pattern(shell_info, max_shells=MAX_SHELLS):
    counts = [str(s["count"]) for s in shell_info]
    n_species = [str(s["n_species"]) for s in shell_info]
    while len(counts) < max_shells:
        counts.append("NA")
        n_species.append("NA")
    return "|".join(counts), "|".join(n_species)


def process_dataframe(df: pd.DataFrame, structure_col: str, base_dir: Path):
    all_rows = []
    temp_files = []
    total = len(df)
    for i, row in df.iterrows():
        try:
            cell = row[structure_col]
            structure = parse_structure_cell(cell, base_dir)
        except Exception as e:
            print(f"[WARN] Skipping input row {i} — could not parse structure: {e}")
            continue

        # iterate every site (optionally: sample or only one representative per element)
        for site_idx in range(len(structure)):
            try:
                shell_info = extract_environment(structure, site_idx)
                counts_pat, nspecies_pat = build_neighbor_pattern(shell_info)
                out = {
                    "_input_row": int(i),
                    "candidate_formula": row.get("formula", row.get("candidate_formula", "")),
                    "prototype_id": row.get("prototype_id", ""),
                    "substitution": row.get("substitution", ""),
                    "wyckoff_letter": row.get("w_letter", ""),
                    "central_site_index": int(site_idx),
                    "central_species": structure[site_idx].species_string,
                    "central_frac_coords": ",".join(f"{x:.6f}" for x in structure[site_idx].frac_coords),
                    "neighbor_counts_pattern": counts_pat,
                    "neighbor_nspecies_pattern": nspecies_pat,
                    "n_detected_shells": len(shell_info)
                }
                # raw shell columns
                for si, sh in enumerate(shell_info, start=1):
                    out[f"shell{si}_count"] = sh["count"]
                    out[f"shell{si}_nspecies"] = sh["n_species"]
                    out[f"shell{si}_species"] = sh["species"]
                    out[f"shell{si}_distances_A"] = ";".join(f"{d:.4f}" for d in sh["distances"])
                all_rows.append(out)
            except Exception as e:
                print(f"[WARN] Failed extracting environment for row {i} site {site_idx}: {e}")
                continue

    return pd.DataFrame(all_rows)


def main(input_csv_path: str = INPUT_CSV, output_csv_path: str = OUTPUT_CSV):
    base_dir = Path.cwd()
    csv_path = Path(input_csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Input CSV not found: {csv_path}")

    df = pd.read_csv(csv_path)
    if df.empty:
        raise ValueError("Input CSV is empty.")

    struct_col = detect_structure_column(df)
    print(f"Detected structure column: '{struct_col}'")

    out_df = process_dataframe(df, struct_col, base_dir)

    if out_df.empty:
        print("No structures processed successfully. Output CSV will not be written.")
        return

    out_df.to_csv(output_csv_path, index=False)
    print(f"Saved output to {output_csv_path} with {len(out_df)} rows.")


if __name__ == "__main__":
    # If running in notebook, change INPUT_CSV variable above or call main(...) manually.
    main()

Detected structure column: 'structure'
Saved output to candidate_neighbor_configurations_from_heas.csv with 800 rows.


In [29]:
df2 = pd.read_csv("candidate_neighbor_configurations_from_heas.csv")
df2

,_input_row,candidate_formula,prototype_id,substitution,wyckoff_letter,central_site_index,central_species,central_frac_coords,neighbor_counts_pattern,neighbor_nspecies_pattern,...,shell2_species,shell2_distances_A,shell3_count,shell3_nspecies,shell3_species,shell3_distances_A,shell4_count,shell4_nspecies,shell4_species,shell4_distances_A
0,0,NaN,NaN,NaN,NaN,0,Al,"0.000000,0.245434,0.259066",1|2|14|24,1|1|4|4,...,Co,1.5118;1.5118,14,4,"Co,Fe,Ni,Si",1.6411;1.6411;1.7335;1.7335;1.7451;1.7451;1.81...,24,4,"Co,Fe,Ni,Si",2.3839;2.3839;2.3839;2.4508;2.4508;2.4508;2.45...
1,0,NaN,NaN,NaN,NaN,1,Al,"1.000000,0.746860,0.748368",1|16|24|8,1|3|4|3,...,"Co,Fe,Ni",1.7420;1.7420;1.7492;1.7492;1.7622;1.7622;1.76...,24,4,"Co,Fe,Ni,Si",2.4159;2.4159;2.4159;2.4215;2.4215;2.4215;2.43...,8,3,"Co,Fe,Ni",3.4981;3.4981;3.5018;3.5018;3.5129;3.5129;3.52...
2,0,NaN,NaN,NaN,NaN,2,Si,"2.000000,0.513840,0.999084",1|16|24|8,1|3|5|2,...,"Co,Fe,Ni",1.6449;1.6449;1.6718;1.6718;1.7436;1.7436;1.74...,24,5,"Al,Co,Fe,Ni,Si",2.4118;2.4118;2.4118;2.4159;2.4159;2.4159;2.44...,8,2,"Fe,Ni",3.4322;3.4322;3.4607;3.4607;3.5167;3.5167;3.52...
3,0,NaN,NaN,NaN,NaN,3,Si,"3.000000,0.972854,0.250913",1|4|10|2,1|1|4|1,...,Ni,1.5637;1.5637;1.5872;1.5872,10,4,"Al,Co,Fe,Ni",1.7143;1.7143;1.7255;1.7255;1.7636;1.7636;1.82...,2,1,Si,2.0686;2.0686
4,0,NaN,NaN,NaN,NaN,4,Si,"4.000000,0.266093,0.253829",1|14|2|24,1|3|1|4,...,"Co,Fe,Ni",1.5576;1.5576;1.6107;1.6107;1.6684;1.6684;1.72...,2,1,Si,2.0686;2.0686,24,4,"Co,Fe,Ni,Si",2.3755;2.3755;2.3755;2.4236;2.4236;2.4236;2.42...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
795,24,NaN,NaN,NaN,NaN,27,Co,"27.000000,0.752904,-0.004453",1|18|21|2,1|5|4|1,...,"Co,Fe,Mn,Ni,Si",1.6746;1.6746;1.7055;1.7055;1.7408;1.7408;1.74...,21,4,"Co,Mn,Ni,Si",2.4516;2.4516;2.4516;2.4785;2.4785;2.4785;2.48...,2,1,Si,3.3614;3.3614
796,24,NaN,NaN,NaN,NaN,28,Co,"28.000000,0.753022,0.495377",1|16|21|8,1|4|4|4,...,"Co,Fe,Ni,Si",1.6841;1.6841;1.7015;1.7015;1.7050;1.7050;1.75...,21,4,"Co,Mn,Ni,Si",2.4022;2.4022;2.4022;2.4655;2.4655;2.4655;2.47...,8,4,"Co,Fe,Mn,Si",3.4644;3.4644;3.4923;3.4923;3.5669;3.5669;3.58...
797,24,NaN,NaN,NaN,NaN,29,Co,"29.000000,0.249749,0.257320",1|16|27|4,1|5|4|2,...,"Co,Fe,Mn,Ni,Si",1.6799;1.6799;1.7558;1.7558;1.7635;1.7635;1.76...,27,4,"Co,Fe,Mn,Ni",2.4228;2.4228;2.4228;2.4474;2.4474;2.4474;2.50...,4,2,"Co,Ni",3.4301;3.4301;3.4532;3.4532
798,24,NaN,NaN,NaN,NaN,30,Co,"30.000000,0.251745,0.748252",1|14|27|4,1|4|4|2,...,"Fe,Mn,Ni,Si",1.6743;1.6743;1.7503;1.7503;1.7724;1.7724;1.77...,27,4,"Co,Fe,Mn,Ni",2.4532;2.4532;2.4532;2.4670;2.4670;2.4670;2.48...,4,2,"Co,Fe",3.4301;3.4301;3.4580;3.4580


In [31]:
import pandas as pd

# load original dataset
df_original = pd.read_csv("hea_dataset.csv")

# load neighbor output
df_neighbors = pd.read_csv("candidate_neighbor_configurations_from_heas.csv")

# create dictionary: row_index -> formula
formula_dict = df_original["composition"].to_dict()

# map formula into neighbor dataframe
df_neighbors["candidate_formula"] = df_neighbors["_input_row"].map(formula_dict)

# move formula column to front (optional)
cols = ["candidate_formula"] + [c for c in df_neighbors.columns if c != "candidate_formula"]
df_neighbors = df_neighbors[cols]

# save corrected csv
df_neighbors.to_csv("candidate_neighbor_configurations_with_formula.csv", index=False)

print("Done.")

Done.
